# Bone Fracture Segmentation: SAM 3 vs U-Net

This notebook demonstrates a **Teacher-Student Distillation** workflow:
1.  **SAM 3 (Teacher)**: Generates "Ground Truth" masks from unlabeled X-ray images (using Zero-Shot capabilities).
2.  **U-Net (Student)**: Trains on these generated masks to learn the segmentation task.
3.  **Comparison**: Evaluate both models on Speed (Inference Time) and Visual Quality.

In [ ]:
# Install dependencies (Run if needed)
# !pip install torch torchvision torchaudio segment-anything-2 ultralytics segmentation-models-pytorch albumentations matplotlib opencv-python
# !pip install -e temp_sam3  # Install local SAM 3 clone

In [ ]:
import os
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import glob
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from PIL import Image
from tqdm import tqdm

# Check Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Teacher: SAM 3 Auto-Labeling
We use SAM 3 to generate masks for our dataset. Since we don't have box prompts, we will try using a generic text prompt "bone fracture".

In [ ]:
# Load SAM 3
try:
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor
    
    sam3_model = build_sam3_image_model(device=DEVICE)
    sam3_processor = Sam3Processor(sam3_model)
    print("SAM 3 Loaded Successfully!")
except ImportError:
    print("Error: SAM 3 not found. Please install it first.")
except Exception as e:
    print(f"Error loading SAM 3: {e}")

In [ ]:
# Configuration
DATA_DIR = "Bone_Fracture_Binary_Classification"
MASK_DIR = "masks_sam3"
IMG_SIZE = 512  # Resize for faster U-Net training
SAMPLE_SIZE = 50 # Number of images to process for this demo

os.makedirs(MASK_DIR, exist_ok=True)

# Find images
image_paths = glob.glob(os.path.join(DATA_DIR, "**", "*.jpg"), recursive=True)
image_paths += glob.glob(os.path.join(DATA_DIR, "**", "*.png"), recursive=True)
image_paths = sorted(image_paths)[:SAMPLE_SIZE]  # Limit sample size for demo

print(f"Processing {len(image_paths)} images for training...")

In [ ]:
def generate_sam3_mask(processor, image_path):
    image = Image.open(image_path).convert("RGB")
    inference_state = processor.set_image(image)
    
    # Text prompt for SAM 3
    output = processor.set_text_prompt(state=inference_state, prompt="bone fracture")
    
    masks = output["masks"]
    scores = output["scores"]
    
    if len(masks) > 0:
        best_idx = torch.argmax(scores)
        best_mask = masks[best_idx].cpu().numpy().astype(np.uint8) * 255
        return best_mask
    return np.zeros((image.size[1], image.size[0]), dtype=np.uint8)

# Generate Masks
valid_data = [] # Store (img_path, mask_path)

print("Generating Masks with SAM 3...")
for img_path in tqdm(image_paths):
    save_name = os.path.basename(img_path).replace(".jpg", ".png").replace(".jpeg", ".png")
    save_path = os.path.join(MASK_DIR, save_name)
    
    if os.path.exists(save_path):
        valid_data.append((img_path, save_path))
        continue
        
    mask = generate_sam3_mask(sam3_processor, img_path)
    if mask is not None:
        cv2.imwrite(save_path, mask)
        valid_data.append((img_path, save_path))

## 2. Student: Train U-Net
Now we train a U-Net model on the masks generated by SAM 3.

In [ ]:
class BoneDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, mask_path = self.data_list[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = np.where(mask > 127, 1.0, 0.0)
        mask = np.expand_dims(mask, axis=0)

        # Simple normalization
        image = image / 255.0
        image = np.transpose(image, (2, 0, 1)).astype(np.float32)
        mask = mask.astype(np.float32)

        return torch.tensor(image), torch.tensor(mask)

# Create Dataset & Loader
train_ds = BoneDataset(valid_data)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

In [ ]:
# Initialize U-Net
unet = smp.Unet(
    encoder_name="resnet34", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1
)
unet.to(DEVICE)

optimizer = torch.optim.Adam(unet.parameters(), lr=1e-4)
loss_fn = smp.losses.DiceLoss(mode='binary')

# Training Loop
EPOCHS = 10
print("Starting U-Net Training...")

unet.train()
loss_history = []

for epoch in range(EPOCHS):
    epoch_loss = 0
    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = unet(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

# Plot Loss
plt.plot(loss_history)
plt.title("U-Net Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Dice Loss")
plt.show()

## 3. Comparison & Visualization
We compare the "Teacher" (SAM 3) vs the "Student" (U-Net) on a test image.

In [ ]:
def compare_models(img_path):
    image = Image.open(img_path).convert("RGB")
    
    # --- 1. SAM 3 Inference ---
    start_sam = time.time()
    inference_state = sam3_processor.set_image(image)
    out_sam = sam3_processor.set_text_prompt(state=inference_state, prompt="bone fracture")
    sam_time = time.time() - start_sam
    
    # Get best mask
    if len(out_sam["masks"]) > 0:
        best_idx = torch.argmax(out_sam["scores"])
        mask_sam = out_sam["masks"][best_idx].cpu().numpy()
    else:
        mask_sam = np.zeros((image.size[1], image.size[0]))
        
    # --- 2. U-Net Inference ---
    # Preprocess
    unet_input = cv2.resize(np.array(image), (IMG_SIZE, IMG_SIZE))
    unet_input = unet_input / 255.0
    unet_input = np.transpose(unet_input, (2, 0, 1)).astype(np.float32)
    unet_input = torch.tensor(unet_input).unsqueeze(0).to(DEVICE)
    
    start_unet = time.time()
    unet.eval()
    with torch.no_grad():
        out_unet = unet(unet_input)
        out_unet = torch.sigmoid(out_unet).cpu().numpy()[0, 0]
    unet_time = time.time() - start_unet
    
    # --- Visualization ---
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    
    ax[0].imshow(image)
    ax[0].set_title("Input Image")
    
    ax[1].imshow(mask_sam, cmap='gray')
    ax[1].set_title(f"SAM 3 (Teacher)\nTime: {sam_time:.4f}s")
    
    ax[2].imshow(out_unet, cmap='gray')
    ax[2].set_title(f"U-Net (Student)\nTime: {unet_time:.4f}s")
    
    plt.show()
    
    return sam_time, unet_time

# Run Comparison on a random validation image
import random
test_img = random.choice(image_paths)
t_sam, t_unet = compare_models(test_img)

print(f"Speedup: U-Net is {t_sam / t_unet:.1f}x faster than SAM 3")